# Assignment 4 - Generative Models with GANs on Adult Tabular Data

Students:
- Yehonatan Segal, 209359801
- Omer Onn, 318910759

This notebook serves as the main entry point for the assignment pipeline.

The implementation itself is organized as Python modules under `src/` and executable scripts under `scripts/`.  
Each script saves its outputs under `results/`, and trained model artifacts / training plots are saved under `models/`.

The notebook runs the scripts in the same order used to generate the results reported in the PDF report.

## Project structure

- `data/raw/adult.arff` — original Adult dataset.
- `src/` — reusable Python modules.
- `scripts/` — executable scripts for each assignment stage.
- `results/` — CSV/TXT/PNG outputs used in the report.
- `models/` — trained model checkpoints and training-curve plots.
- `Assignment4_Report.pdf` — final report.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
print("Current working directory:", PROJECT_ROOT)
print("Python executable:", sys.executable)

Current working directory: c:\Users\יהונתן סגל\Projects\M.Sc - SISE\First year\Semester B\Deep learning\Works\Deep_Learning_Assignment\Assignment4
Python executable: c:\Users\יהונתן סגל\Projects\M.Sc - SISE\First year\Semester B\Deep learning\Works\Deep_Learning_Assignment\.venv\Scripts\python.exe


## Step 1 - Dataset inspection

Loads the Adult ARFF file, identifies numerical/categorical columns, checks missing values, and saves dataset summaries.

In [2]:
!python scripts/01_inspect_dataset.py

STEP 1: Adult ARFF dataset inspection

Loading dataset from:
C:\Users\יהונתן סגל\Projects\M.Sc - SISE\First year\Semester B\Deep learning\Works\Deep_Learning_Assignment\Assignment4\data\raw\adult.arff

Dataset loaded successfully.
Number of rows: 32561
Number of columns: 15

Columns:
- age
- workclass
- fnlwgt
- education
- education-num
- marital-status
- occupation
- relationship
- race
- sex
- capital-gain
- capital-loss
- hours-per-week
- native-country
- income

Numeric feature columns:
- age
- fnlwgt
- education-num
- capital-gain
- capital-loss
- hours-per-week

Categorical feature columns:
- workclass
- education
- marital-status
- occupation
- relationship
- race
- sex
- native-country

Target column: income

Target distribution:
        count    ratio
income                
<=50K   24720  0.75919
>50K     7841  0.24081

Missing values per column:
age                  0
workclass         1836
fnlwgt               0
education            0
education-num        0
marital-status  

## Step 2 - Stratified train-test splits

Creates three 80%/20% train-test splits using seeds 42, 123, and 2026, while preserving the label ratio of `income`.

In [3]:
!python scripts/02_create_splits.py

STEP 2: Stratified 80/20 train-test splits

Loading dataset...
Dataset path: C:\Users\יהונתן סגל\Projects\M.Sc - SISE\First year\Semester B\Deep learning\Works\Deep_Learning_Assignment\Assignment4\data\raw\adult.arff

Dataset loaded successfully.
Full dataset shape: (32561, 15)
Target column: income
Test size: 0.2
Train size: 0.8
Random seeds: [42, 123, 2026]

Full dataset target distribution:
   seed dataset  label  count    ratio
0    -1    full  <=50K  24720  0.75919
1    -1    full   >50K   7841  0.24081

--------------------------------------------------------------------------------
Creating split for seed: 42
--------------------------------------------------------------------------------
Train shape: (26048, 15)
Test shape: (6513, 15)

Train target distribution:
   seed dataset  label  count     ratio
0    42   train  <=50K  19775  0.759175
1    42   train   >50K   6273  0.240825

Test target distribution:
   seed dataset  label  count     ratio
0    42    test  <=50K   4945  0

## Step 3 - Preprocessing

Applies median imputation and MinMax scaling to numerical features, and one-hot encoding to categorical features and the target.

In [4]:
!python scripts/03_preprocess_splits.py

STEP 3: Preprocessing train/test splits

Loading original ARFF metadata...

Preprocessing decisions:
- Numeric features: median imputation + MinMax scaling to [-1, 1]
- Categorical features: missing values replaced with 'Missing' + one-hot encoding
- Target feature: one-hot encoding
- Encoders use ARFF-declared categories to keep dimensions stable across seeds

Numeric columns:
- age
- fnlwgt
- education-num
- capital-gain
- capital-loss
- hours-per-week

Categorical columns:
- workclass: 9 categories
- education: 17 categories
- marital-status: 8 categories
- occupation: 15 categories
- relationship: 7 categories
- race: 6 categories
- sex: 3 categories
- native-country: 42 categories

Target categories:
['<=50K', '>50K']

--------------------------------------------------------------------------------
Preprocessing split for seed: 42
--------------------------------------------------------------------------------
Raw train shape: (26048, 15)
Raw test shape: (6513, 15)

Processed shap

## Step 4 - Standard GAN training

Trains the standard unconditional GAN for the three seeds and generates synthetic datasets equal in size to the training sets.

In [5]:
!python scripts/04_train_gan.py

STEP 4: Train standard GAN

GAN configuration:
- Latent dim: 64
- Batch size: 256
- Epochs: 300
- Learning rate: 0.0002
- Adam betas: (0.5, 0.999)
- Seeds: [42, 123, 2026]

--------------------------------------------------------------------------------
Training standard GAN for seed: 42
--------------------------------------------------------------------------------
Loaded train GAN data from: C:\Users\יהונתן סגל\Projects\M.Sc - SISE\First year\Semester B\Deep learning\Works\Deep_Learning_Assignment\Assignment4\data\processed\preprocessed\seed_42\train_gan.npy
Train GAN shape: (26048, 115)
Numeric dim: 6
GAN data dim: 115
Using device: cpu
GPU is not available. Training will run on CPU.

Model dimensions:
- GAN data dimension: 115
- Numeric dimension: 6
- Non-numeric dimension: 109
- Latent dimension: 64
- Batch size: 256
- Epochs: 300
- Learning rate: 0.0002
Epoch 0001/300 | D loss: 0.6799 | G loss: 1.7128 | D(real): 0.7144 | D(fake): 0.2506
Epoch 0010/300 | D loss: 0.4032 | G loss: 


Training GAN: 100%|██████████| 300/300 [11:46<00:00,  2.35s/it, D_loss=0.425, G_loss=2.92]

Training GAN: 100%|██████████| 300/300 [12:38<00:00,  2.53s/it, D_loss=0.379, G_loss=3.08]

Training GAN: 100%|██████████| 300/300 [15:23<00:00,  3.08s/it, D_loss=0.402, G_loss=3.01]


## Step 5 - Conditional GAN training

Trains the conditional GAN. The generator receives noise concatenated with a requested one-hot income label, and the discriminator receives samples together with the same condition.

In [6]:
!python scripts/05_train_cgan.py

STEP 5: Train conditional GAN

cGAN configuration:
- Latent dim: 64
- Batch size: 256
- Epochs: 300
- Learning rate: 0.0002
- Adam betas: (0.5, 0.999)
- Seeds: [42, 123, 2026]

--------------------------------------------------------------------------------
Training conditional GAN for seed: 42
--------------------------------------------------------------------------------
Loaded train features from: C:\Users\יהונתן סגל\Projects\M.Sc - SISE\First year\Semester B\Deep learning\Works\Deep_Learning_Assignment\Assignment4\data\processed\preprocessed\seed_42\train_features.npy
Loaded train conditions from: C:\Users\יהונתן סגל\Projects\M.Sc - SISE\First year\Semester B\Deep learning\Works\Deep_Learning_Assignment\Assignment4\data\processed\preprocessed\seed_42\train_target_onehot.npy
Train features shape: (26048, 113)
Train conditions shape: (26048, 2)
Numeric dim: 6
Feature dim: 113
Condition dim: 2

Training condition distribution:
- income__<=50K: count=19775, ratio=0.759175
- income__>5


Training cGAN: 100%|██████████| 300/300 [19:20<00:00,  3.87s/it, D_loss=0.463, G_loss=2.99]

Training cGAN: 100%|██████████| 300/300 [22:32<00:00,  4.51s/it, D_loss=0.419, G_loss=3.26]

Training cGAN: 100%|██████████| 300/300 [16:53<00:00,  3.38s/it, D_loss=0.484, G_loss=3.13]


## Step 6 - Main evaluation

Computes Detection AUC and Efficacy using Random Forest classifiers. Also saves real-vs-synthetic visualizations.

In [7]:
!python scripts/06_evaluate_models.py

STEP 6: Evaluation and reported results

Evaluation configuration:
- Random Forest n_estimators: 200
- Random Forest max_depth: None
- Random Forest n_jobs: -1
- Detection folds: 4
- Seeds: [42, 123, 2026]

Loading data for seed 42

Evaluating GAN for seed 42
--------------------------------------------------------------------------------
Real train GAN shape: (26048, 115)
Synthetic GAN shape: (26048, 115)
Real train features shape: (26048, 113)
Synthetic features shape: (26048, 113)
Real test features shape: (6513, 113)
Synthetic categorical and target blocks were hard-decoded before evaluation.

Detection metric:
AUC mean: 0.999999
AUC std : 0.000001
Note: for detection, lower is better; 0.5 is ideal.

Efficacy metric:
Real-train AUC      : 0.907381
Synthetic-train AUC : 0.838122
Efficacy ratio      : 0.923671
Note: for efficacy, higher is better; 1.0 is ideal.

Evaluating cGAN for seed 42
--------------------------------------------------------------------------------
Real train GAN

## Step 7 - Discrete feature generation

Compares two differentiable strategies for generating categorical features:
1. Softmax distributions passed directly to the discriminator.
2. Gumbel-Softmax with a straight-through estimator.

In [8]:
!python scripts/07_discrete_feature_experiments.py

STEP 7: Discrete feature generation experiments

Experiment setup:
- Seed: 42
- Train GAN shape: (26048, 115)
- Numeric dim: 6
- Feature dim: 113
- Target dim: 2
- Number of categorical blocks including target: 9
- Epochs: 300
- Strategies: softmax, gumbel_st

Categorical blocks:
- workclass: 9
- education: 17
- marital-status: 8
- occupation: 15
- relationship: 7
- race: 6
- sex: 3
- native-country: 42
- income: 2

Real categorical entropy:
            block  ...  real_entropy_ratio_of_uniform
0       workclass  ...                       0.522526
1       education  ...                       0.719067
2  marital-status  ...                       0.611988
3      occupation  ...                       0.900898
4    relationship  ...                       0.766075
5            race  ...                       0.307665
6             sex  ...                       0.577678
7  native-country  ...                       0.174729
8          income  ...                       0.796409

[9 rows x 5 c


Training softmax: 100%|██████████| 300/300 [12:53<00:00,  2.58s/it, D_loss=0.649, G_loss=2.53]

Training gumbel_st: 100%|██████████| 300/300 [15:27<00:00,  3.09s/it, D_loss=0.237, G_loss=16.4]


## Step 8 - Mode collapse

Deliberately induces mode collapse, measures it using quantitative indicators, and tests a diversity-penalty mitigation.

In [9]:
!python scripts/08_mode_collapse.py

STEP 8: Mode collapse - induce, diagnose, and mitigate

Experiment setup:
- Seed: 42
- Train GAN shape: (26048, 115)
- Numeric dim: 6
- Epochs: 150
- Collapsed latent dim: 4
- Collapsed generator hidden dims: (32,)
- Collapsed discriminator hidden dims: (256, 128, 64)
- D updates per G update: 5
- Mitigation diversity weight: 0.5

Running experiment: collapsed
Using device: cpu
GPU is not available. Training will run on CPU.

Training configuration:
- Data dim: 115
- Numeric dim: 6
- Latent dim: 4
- Generator hidden dims: (32,)
- Discriminator hidden dims: (256, 128, 64)
- D updates per G update: 5
- Diversity weight: 0.0
- Epochs: 150
- Batch size: 256
Epoch 0001/150 | D loss: 0.0935 | G total: 7.4792 | G adv: 7.4792 | Div: 0.1014 | D(real): 0.9551 | D(fake): 0.0291
Epoch 0010/150 | D loss: 0.0034 | G total: 10.3399 | G adv: 10.3399 | Div: 0.0643 | D(real): 0.9993 | D(fake): 0.0014
Epoch 0020/150 | D loss: 0.0070 | G total: 10.9980 | G adv: 10.9980 | Div: 0.0650 | D(real): 0.9984 | D(


Training mode-collapse experiment: 100%|██████████| 150/150 [18:00<00:00,  7.20s/it, D_loss=0.0572, G_loss=5.9]

Training mode-collapse experiment: 100%|██████████| 150/150 [1:13:56<00:00, 29.58s/it, D_loss=0.0694, G_loss=4.87]


## Step 9 - Open-ended contribution

Tests a Feature Matching GAN and compares it to the unmodified baseline GAN.

In [ ]:
!python scripts/09_open_ended_feature_matching.py

STEP 9: Open-ended contribution - Feature Matching GAN

Experiment setup:
- Seed: 42
- Train GAN shape: (26048, 115)
- Numeric dim: 6
- Feature dim: 113
- Target dim: 2
- Epochs: 300
- Feature matching weight: 10.0
Using device: cpu
GPU is not available. Training will run on CPU.

Feature Matching GAN configuration:
- Data dim: 115
- Numeric dim: 6
- Latent dim: 64
- Batch size: 256
- Epochs: 300
- Learning rate: 0.0002
- Feature matching weight: 10.0
Epoch 0001/300 | D loss: 0.7291 | G total: 4.1723 | G adv: 1.6220 | FM: 0.2550 | D(real): 0.7006 | D(fake): 0.2689
Epoch 0010/300 | D loss: 0.4282 | G total: 8.5723 | G adv: 2.6193 | FM: 0.5953 | D(real): 0.8706 | D(fake): 0.1349
Epoch 0020/300 | D loss: 0.4888 | G total: 8.2976 | G adv: 2.3134 | FM: 0.5984 | D(real): 0.8508 | D(fake): 0.1535
Epoch 0030/300 | D loss: 0.5141 | G total: 10.1235 | G adv: 2.2081 | FM: 0.7915 | D(real): 0.8388 | D(fake): 0.1629
Epoch 0040/300 | D loss: 0.5021 | G total: 12.9924 | G adv: 2.2766 | FM: 1.0716 | D


Training Feature Matching GAN: 100%|██████████| 300/300 [20:59<00:00,  4.20s/it, D_loss=0.457, G_loss=60.7]


: 

## Notes

The full outputs used in the report are saved under `results/`.  
The figures used in the appendix are saved under `models/` and `results/`.

Some scripts may take time to run because they train GANs and Random Forest models.  
The submitted ZIP already includes the generated results and plots.